# Module 4

## Prerequisites

In [1]:
import boto3
import json
import time
from datetime import datetime
import os

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

## Inference profile

In [2]:
prompt = """
What design patterns are used in this code?

You are an expert Python developer. Here's a codebase context:

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = []
    
    def load_data(self, source):
        # Complex data loading logic
        pass
    
    def transform_data(self, transformations):
        # Data transformation pipeline
        pass
    
    def validate_data(self, rules):
        # Data validation logic
        pass

This class is part of a larger system with 50+ similar classes.
Always reference this context when answering questions.
"""

### Invoking cross-region inference profile

In [5]:
response = bedrock.converse(
    modelId='global.amazon.nova-2-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "2daeb0f6-e221-4d77-aea3-8746a8b45845",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Thu, 08 Oct 2026 19:19:06 GMT",
      "content-type": "application/json",
      "content-length": "5109",
      "connection": "keep-alive",
      "x-amzn-requestid": "2daeb0f6-e221-4d77-aea3-8746a8b45845"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Based on the provided code snippet and context, several design patterns are evident or could be applied to this codebase. Here's a breakdown of the patterns **already present** and those **commonly used** in such a scenario:\n\n---\n\n### \u2705 **Existing Patterns in the Code**\n\n#### 1. **Template Method Pattern**\n- **Evidence**: The `DataProcessor` class defines a skeleton of an algorithm (`load_data`, `transform_data`, `validate_data`) in a method (though not explicitly shown), but **defers some s

### Use your own inference profile

In [6]:
# create an inference profile
bedrock_control = boto3.client('bedrock', region_name='us-east-1')

response = bedrock_control.create_inference_profile(
    inferenceProfileName='demo-nova-2-lite-profile',
    description='Custom inference profile for Nova 2 Lite',
    modelSource={
        'copyFrom': 'arn:aws:bedrock:us-east-1:767397992380:inference-profile/global.amazon.nova-2-lite-v1:0'
    }
)
inference_profile_arn = response['inferenceProfileArn']

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "dc849c7a-db2b-4c00-8eb9-f68e882eb06c",
    "HTTPStatusCode": 201,
    "HTTPHeaders": {
      "date": "Thu, 08 Oct 2026 19:23:12 GMT",
      "content-type": "application/json",
      "content-length": "125",
      "connection": "keep-alive",
      "x-amzn-requestid": "dc849c7a-db2b-4c00-8eb9-f68e882eb06c"
    },
    "RetryAttempts": 0
  },
  "inferenceProfileArn": "arn:aws:bedrock:us-east-1:767397992380:application-inference-profile/kxvoxogavhkk",
  "status": "ACTIVE"
}


In [5]:
response = bedrock.converse(
    modelId=inference_profile_arn,
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "c6148741-a41b-4ef8-86da-d35b73eae383",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Tue, 06 Oct 2026 19:44:06 GMT",
      "content-type": "application/json",
      "content-length": "5432",
      "connection": "keep-alive",
      "x-amzn-requestid": "c6148741-a41b-4ef8-86da-d35b73eae383"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Based on the provided code snippet for the `DataProcessor` class, we can analyze the design patterns that might be used in the broader system, considering that there are 50+ similar classes. Here are some potential design patterns that may be applicable:\n\n### 1. **Factory Pattern**\nSince there are many similar classes, a Factory Pattern might be used to instantiate the correct class based on some input parameters or configuration.\n\n```python\nclass DataProcessorFactory:\n    @staticmethod\n    def 

In [8]:
response = bedrock_control.delete_inference_profile(
    inferenceProfileIdentifier=inference_profile_arn
)

## Batch Inference

Upload jsonl file to S3

In [7]:
bucket_name = "batch-data-inference-dk1006"

s3 = boto3.client('s3')

# Upload manifest to S3
input_key = 'input/city-reviews-manifest.jsonl'
s3.upload_file('input/city-reviews-manifest.jsonl', bucket_name, input_key)

Call the CreateModelInvocationJob API

In [8]:
accountnumber = "767397992380"

# Create the bedrock control plane client
bedrock_control = boto3.client('bedrock', region_name='us-east-1')

# Create batch inference job
job_name = f"batch-job-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

response = bedrock_control.create_model_invocation_job(
    jobName=job_name,
    roleArn=f"arn:aws:iam::{accountnumber}:role/BedrockBatchRole",
    modelId="us.amazon.nova-lite-v1:0",
    inputDataConfig={
        's3InputDataConfig': {
            's3Uri': f's3://{bucket_name}/input/'
        }
    },
    outputDataConfig={
        's3OutputDataConfig': {
            's3Uri': f's3://{bucket_name}/output/'
        }
    }
)

job_arn = response['jobArn']


Get the Job Status

In [9]:
status_response = bedrock_control.get_model_invocation_job(jobIdentifier=job_arn)
status = status_response['status']
print(f"Job status: {status}")

Job status: Submitted


Instead of waiting, let's use a jobArn for a job we know already finished

In [10]:
completed_job_arn = job_arn

Wait for the job to finish

In [11]:
# Wait for job completion
while True:
    status_response = bedrock_control.get_model_invocation_job(jobIdentifier=completed_job_arn)
    status = status_response['status']
    print(f"Job status: {status}")
    
    if status in ['Completed', 'PartiallyCompleted', 'Failed', 'Stopped', 'Expired']:
        break
    
    time.sleep(30)



Job status: Submitted
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Validating
Job status: Scheduled
Job status: Scheduled
Job status: Scheduled
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: InProgress
Job status: Completed


Download the result

In [13]:
if status in ["Completed", "PartiallyCompleted"]:
    # Download results
    output_objects = s3.list_objects_v2(Bucket=bucket_name, Prefix='output/')
    
    os.makedirs('batch-output', exist_ok=True)
    
    for obj in output_objects.get('Contents', []):
        if obj['Key'].endswith('/'):
            continue
            
        local_file = os.path.join('batch-output', obj['Key'].split('/')[-1])
        s3.download_file(bucket_name, obj['Key'], local_file)
        print(f"Downloaded: {local_file}")


Downloaded: batch-output\city-reviews-manifest.jsonl.out
Downloaded: batch-output\manifest.json.out


Stop the job we created to save on cost

In [ ]:
response = bedrock_control.stop_model_invocation_job(
    jobIdentifier=job_arn
)

## Prompt Caching with Nova Lite

In [12]:
# Large context that we want to cache
cached_context = """
You are a expert Python developer. Here's a large codebase context:

class DataProcessor:
    def __init__(self, config):
        self.config = config
        self.data = []
    
    def load_data(self, source):
        # Complex data loading logic
        pass
    
    def transform_data(self, transformations):
        # Data transformation pipeline
        pass
    
    def validate_data(self, rules):
        # Data validation logic
        pass

This class is part of a larger system with 50+ similar classes.
Always reference this context when answering questions.
"""

First request that establishes the cache

In [19]:
firstRequest = bedrock.converse(
    modelId='amazon.nova-pro-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "text": cached_context
                },
                {
                    "cachePoint": {
                        "type": "default", "ttl": "1h"
                    }
                },
                {
                    "text": "What design patterns are used in this code?"
                }
          ]
      }
    ]
)

print(json.dumps(firstRequest, indent=2))

ValidationException: An error occurred (ValidationException) when calling the Converse operation: Extended TTL prompt caching is only supported for Anthropic models

Second request that uses cached context

In [4]:
secondRequest = bedrock.converse(
    modelId='amazon.nova-lite-v1:0',
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": cached_context
                },
                {
                    "cachePoint": {
                        "type": "default"
                    }
                },
                {
                    "text": "what else can you tell me about this code?"
                }
          ]
      }
    ]
)

print(json.dumps(secondRequest, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "d5584a24-f24b-4098-ac96-d5061ac38a5d",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Fri, 18 Sep 2026 16:36:39 GMT",
      "content-type": "application/json",
      "content-length": "5648",
      "connection": "keep-alive",
      "x-amzn-requestid": "d5584a24-f24b-4098-ac96-d5061ac38a5d"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Given the context provided, here are several insights and recommendations that can be applied to the `DataProcessor` class and the larger system it is part of:\n\n### Class Design and Structure\n\n1. **Initialization and Configuration:**\n   - The `__init__` method initializes the `config` and an empty list `self.data`. This suggests that `config` is a crucial component for the class's operation. It's important to ensure that `config` is properly validated and documented.\n   - Consider adding type hint

Compare the usage

In [5]:
print("### First request ###")
print("Usage:", json.dumps(firstRequest['usage'], indent=2))
print("Metrics:", json.dumps(firstRequest['metrics'], indent=2))

print("\n\n### Second request ###")
print("Usage:", json.dumps(secondRequest['usage'], indent=2))
print("Metrics:", json.dumps(secondRequest['metrics'], indent=2))

### First request ###
Usage: {
  "inputTokens": 9,
  "outputTokens": 1252,
  "totalTokens": 1385,
  "cacheReadInputTokens": 0,
  "cacheWriteInputTokens": 124
}
Metrics: {
  "latencyMs": 7385
}


### Second request ###
Usage: {
  "inputTokens": 10,
  "outputTokens": 1101,
  "totalTokens": 1235,
  "cacheReadInputTokens": 124,
  "cacheWriteInputTokens": 0
}
Metrics: {
  "latencyMs": 5243
}


## Converse on documents
### Using `bytes` as a source

In [ ]:
with open('input/AnyCompany_financial_10K.pdf', "rb") as file:
    doc_bytes = file.read()

In [ ]:
response = bedrock.converse(
    modelId='us.amazon.nova-lite-v1:0',
    messages=[
       {
            "role": "user",
            "content": [
                {
                    "document": {
                        "format": "pdf",
                        "name": "AnyCompany_financial",
                        "source": {
                            "bytes": doc_bytes
                        }
                    }
                },
                {
                    "text": "What investments have AnyCompany made?"
                }
          ]
      }
    ]
)

print(json.dumps(response, indent=2))

### Using s3Location as a source

In [ ]:
bucket_name = "batch-data-inference-dk01"

s3 = boto3.client('s3')

input_key = 'AnyCompany_financial_10K.pdf'
s3.upload_file('input/AnyCompany_financial_10K.pdf', bucket_name, input_key)

## Conversation history

### Create the DynamoDB Table and set the time to live

In [14]:
table_name = "conversation-history-dk01"

dynamodb = boto3.client('dynamodb', region_name="us-east-1")
response = dynamodb.create_table(
    TableName=table_name,
    BillingMode='PAY_PER_REQUEST',
    AttributeDefinitions=[
        {
            'AttributeName': 'userId',
            'AttributeType': 'S'
        },
        {
            'AttributeName': 'timestamp',
            'AttributeType': 'N'
        }
    ],
    KeySchema=[
        {
            'AttributeName': 'userId',
            'KeyType': 'HASH'
        },
        {
            'AttributeName': 'timestamp',
            'KeyType': 'RANGE'
        }
    ]
)

# Wait for table to be active
waiter = dynamodb.get_waiter('table_exists')
waiter.wait(TableName=table_name)

ttl_response = dynamodb.update_time_to_live(
    TableName=table_name,
    TimeToLiveSpecification={
        'AttributeName': 'ttl',
        'Enabled': True
    }
)

### Create the functions to get the conversation history and store new messages

In [15]:
table_name = "conversation-history-dk01"
ddbclient = boto3.client('dynamodb', region_name = "us-east-1")

def get_conversation_history(user_id):
    # Pagination is required in case the conversation is more than 1MB.
    paginator = ddbclient.get_paginator('query')
    pages = paginator.paginate(
        TableName = table_name,
        KeyConditionExpression = 'userId = :val',
        ExpressionAttributeValues = {':val': {'S': user_id}}
    )
    messages = []
    for page in pages:
        for item in page.get('Items', []):
            messages.append({
                "role": item["role"]["S"],
                "content": [{ "text": item["message"]["S"] }]
            })
    return messages

In [22]:
dynamodbResource = boto3.resource('dynamodb', region_name = "us-east-1")
conversation_table = dynamodbResource.Table(table_name)

def store_message(user_id, message, role):
    now_in_seconds = int(time.time())
    expire_ttl = now_in_seconds + (1 * 24 * 60 * 60) # 1 day
    
    conversation_table.put_item(
        Item = {
            'userId': user_id,
            'timestamp': now_in_seconds,
            'message': message,
            'role': role,
            'ttl': expire_ttl
        }
    )


### Create the function to send messages to the LLM

In [23]:
def send_message(user_id, message):
    
    # Load the conversation history
    conversation_messages = get_conversation_history(user_id)

    # Store the new message
    store_message(user_id, message, "user")
    
    # Add the new message to the conversation
    conversation_messages.append({
        "role": "user",
        "content": [{ "text": message }]
    })

    # Send the conversation to the LLM
    model_response = bedrock.converse(
        modelId="amazon.nova-lite-v1:0",
        messages=conversation_messages,
        system = [{ 
            "text": "Please provide a helpful, conversational response based on the available information and conversation history." 
        }],
        inferenceConfig = {
            "maxTokens": 300,
            "temperature": 0.7,
            "topP": 0.9
        }
    )

    # Parse the message from the LLM, store it and return it
    assistant_msg = model_response['output']['message']['content'][0]['text']
    store_message(user_id, assistant_msg, "assistant")

    return conversation_messages, assistant_msg

### Sending messages

In [24]:
user_id = "Disha"
convo, response = send_message(user_id, "My name is Disha and I love to eat pizza. What are some good pizza places near me in Vancouver?")
print(response)


Hello Disha! I'd be happy to help you find some great pizza spots in Vancouver. Here are a few popular and highly-regarded pizza places in the city:

1. **Libretto**: This is a well-known pizzeria in Vancouver, particularly for its wood-fired pizzas. The atmosphere is cozy, and the pizzas are crafted with high-quality ingredients. Be sure to try their classic margherita or one of their seasonal specials.

2. **Viva La Pizza**: Located in the heart of Vancouver, Viva La Pizza is famous for its thin-crust pizzas. They offer a variety of toppings and even vegan options. The atmosphere is lively, making it a great spot for a casual meal with friends.

3. **Il Caminetto**: If you're looking for a more upscale dining experience, Il Caminetto is a fantastic choice. It's renowned for its authentic Neapolitan-style pizzas cooked in a wood-fired oven. The setting is elegant, and the pizzas are simply delicious.

4. **Scaramouche**: This is a bit of a hidden gem located in the West End. It’s know

In [25]:
convo, response = send_message(user_id, "What do I love to eat?")
print(response)

You mentioned earlier that you love to eat pizza. It sounds like pizza is a favorite treat for you! Whether you’re in the mood for a classic margherita, a loaded meat lover’s pie, or something a bit more adventurous, there are plenty of delicious options available in Vancouver to satisfy your pizza cravings. Enjoy your next pizza adventure! If you have any other questions or need more recommendations, feel free to ask.


In [26]:
print(json.dumps(convo, indent=2))

[
  {
    "role": "user",
    "content": [
      {
        "text": "My name is Disha and I love to eat pizza. What are some good pizza places near me in Vancouver?"
      }
    ]
  },
  {
    "role": "assistant",
    "content": [
      {
        "text": "Hello Disha! I'd be happy to help you find some great pizza spots in Vancouver. Here are a few popular and highly-regarded pizza places in the city:\n\n1. **Libretto**: This is a well-known pizzeria in Vancouver, particularly for its wood-fired pizzas. The atmosphere is cozy, and the pizzas are crafted with high-quality ingredients. Be sure to try their classic margherita or one of their seasonal specials.\n\n2. **Viva La Pizza**: Located in the heart of Vancouver, Viva La Pizza is famous for its thin-crust pizzas. They offer a variety of toppings and even vegan options. The atmosphere is lively, making it a great spot for a casual meal with friends.\n\n3. **Il Caminetto**: If you're looking for a more upscale dining experience, Il Cam